# 08 · Condition-blind cNMF program review

This notebook evaluates whether each cNMF program is a coherent state of its source lineage and whether its spatial pattern appears biological rather than driven by spillover, segmentation, tissue edges, or one section.

The review hides condition labels and replaces sample IDs with deterministic section aliases. Select one lineage and program, inspect the evidence panels, and save a decision only after enabling the explicit write switch.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import anndata as ad
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "spatial_workflow").is_dir():
    candidate = REPO_ROOT.parent
    if (candidate / "src" / "spatial_workflow").is_dir():
        REPO_ROOT = candidate
    else:
        raise RuntimeError("Start Jupyter from the spatial-workflow repository")
sys.path.insert(0, str(REPO_ROOT / "src"))

from spatial_workflow.cnmf import usage_columns
from spatial_workflow.cnmf_program_review import (
    build_review_decision,
    high_usage_gene_support,
    high_usage_neighbor_context,
    plot_blinded_spatial_usage,
    prepare_blinded_usage_obs,
    program_usage_qc_summary,
    select_high_usage_cells,
    upsert_review_decision,
)

CNMF_ROOT = REPO_ROOT / "results" / "ab_xenium" / "05_cnmf"
WHITELIST_PATH = (
    CNMF_ROOT / "program_review" / "cnmf_program_whitelist_draft.tsv"
)
METRICS_PATH = (
    CNMF_ROOT
    / "program_review"
    / "cnmf_program_review_metrics_condition_blind.tsv"
)
QUEUE_PATH = (
    CNMF_ROOT / "program_review" / "cnmf_program_review_queue_draft.tsv"
)
DECISION_PATH = (
    CNMF_ROOT / "program_review" / "human_review_decisions_draft.tsv"
)
MASTER_H5AD = (
    CNMF_ROOT
    / "integrated"
    / "ab_xenium_cellcharter_cnmf_selected_draft.h5ad"
)

LINEAGE_DIRECTORIES = {
    "astrocyte": "astrocyte",
    "oligodendrocyte": "oligodendrocyte",
    "microglia": "microglia_cluster_sub_all",
    "inhibitory_neuron": "inhibitory_neuron",
    "excitatory_neuron": "excitatory_neuron",
    "perivascular": "perivascular",
    "epd": "epd",
    "chp": "chp",
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## Program inventory

This section verifies the draft whitelist, its checksum, and the source usage objects. Counts are summarized by lineage and draft decision without exposing condition information.

In [ ]:
whitelist = pd.read_csv(
    WHITELIST_PATH, sep="\t", dtype=str, keep_default_na=False
)
whitelist_sha256 = file_sha256(WHITELIST_PATH)

inventory = (
    whitelist.groupby("lineage", sort=False)
    .agg(
        selected_k=("selected_k", "first"),
        raw_programs=("program", "size"),
        primary_programs=("primary_include", lambda s: int(s.eq("TRUE").sum())),
        sensitivity_total=("sensitivity_include", lambda s: int(s.eq("TRUE").sum())),
        excluded_programs=("decision", lambda s: int(s.eq("exclude").sum())),
    )
    .reset_index()
)
display(inventory)
print("whitelist SHA-256:", whitelist_sha256)

program_metrics = pd.read_csv(METRICS_PATH, sep="\t")
review_queue = pd.read_csv(QUEUE_PATH, sep="\t", keep_default_na=False)
if not program_metrics["metric_condition_blind"].astype(bool).all():
    raise RuntimeError("Program metric table is not marked condition-blind")

review_columns = [
    "review_order", "review_tier", "lineage", "program", "decision", "metric_flags"
]
display(review_queue[review_columns].head(25))

,lineage,selected_k,raw_programs,primary_programs,sensitivity_total,excluded_programs
0,astrocyte,18,18,3,5,13
1,oligodendrocyte,25,25,7,8,17
2,microglia,18,18,3,4,14
3,inhibitory_neuron,19,19,12,12,7
4,excitatory_neuron,25,25,14,18,7
5,perivascular,26,26,9,9,17
6,epd,10,10,4,4,6
7,chp,9,9,2,4,5


whitelist SHA-256: 4ecef413dd3892b2d501afe6ce60b45031225fbf6dfb6b302a7ae947c0b2746c


,review_order,review_tier,lineage,program,decision,metric_flags
0,1,1_individual_attention,astrocyte,Usage_1,include_primary,strong_within_lineage_spatial_clustering
1,2,1_individual_attention,astrocyte,Usage_2,include_primary,strong_within_lineage_spatial_clustering
2,3,1_individual_attention,astrocyte,Usage_7,include_primary,strong_within_lineage_spatial_clustering
3,4,1_individual_attention,astrocyte,Usage_13,include_sensitivity,strong_within_lineage_spatial_clustering
4,5,1_individual_attention,astrocyte,Usage_16,exclude,strong_within_lineage_spatial_clustering
5,6,1_individual_attention,astrocyte,Usage_17,include_sensitivity,weak_raw_detection_of_annotation_genes
6,7,1_individual_attention,oligodendrocyte,Usage_9,exclude,strong_within_lineage_spatial_clustering
7,8,1_individual_attention,oligodendrocyte,Usage_17,include_primary,strong_within_lineage_spatial_clustering
8,9,1_individual_attention,oligodendrocyte,Usage_19,include_primary,
9,10,1_individual_attention,oligodendrocyte,Usage_20,include_primary,


## Select a program for review

Choose one lineage–program pair and the fraction of highest-usage cells to inspect. The notebook uses an exact ranked-cell count and a configurable neighbor radius for contamination screening.

Run the remaining sections after changing these controls; all displays remain condition blind.

In [273]:
REVIEW_ORDER = int(review_queue["review_order"].min())
matches = review_queue.loc[review_queue["review_order"].eq(REVIEW_ORDER)]
if len(matches) != 1:
    raise RuntimeError(f"Expected one review item for order {REVIEW_ORDER}; found {len(matches)}")
review_item = matches.iloc[0]
LINEAGE = str(review_item["lineage"])
PROGRAM = str(review_item["program"])

TOP_FRACTION = 0.05
NEIGHBOR_RADIUS_UM = 30.0
MAX_CONTEXT_FOCAL_CELLS = 2_000
MAX_CORRELATION_CELLS = 50_000

# None selects the blinded section containing the most high-usage cells.
SPATIAL_SECTION = None
SPATIAL_POINT_SIZE = 3.0

display(review_item.to_frame("value"))
print(f"Reviewing {REVIEW_ORDER}/{len(review_queue)}: {LINEAGE} / {PROGRAM}")

,value
review_order,41
review_tier,1_individual_attention
review_reason,new_or_ambiguous_program
lineage,excitatory_neuron
program,Usage_11
proposed_label,RELN superficial excitatory subtype
category,identity
confidence,medium
primary_include,True
sensitivity_include,True


Reviewing 41/150: excitatory_neuron / Usage_11


In [274]:
if LINEAGE not in LINEAGE_DIRECTORIES:
    raise KeyError(f"Unknown lineage {LINEAGE!r}: {list(LINEAGE_DIRECTORIES)}")

lineage_root = CNMF_ROOT / LINEAGE_DIRECTORIES[LINEAGE]
manifest = json.loads((lineage_root / "run_manifest.json").read_text())
usage_adata = ad.read_h5ad(Path(manifest["paths"]["usage_h5ad"]), backed="r")
programs = usage_columns(usage_adata)
if PROGRAM not in programs:
    raise KeyError(f"{PROGRAM!r} is unavailable; choose one of {programs}")

review_obs, sample_aliases = prepare_blinded_usage_obs(
    usage_adata.obs, sample_key="sample_id", usage_cols=programs
)
assert "condition" not in review_obs
assert "sample_id" not in review_obs

program_row = whitelist.loc[
    whitelist["lineage"].eq(LINEAGE) & whitelist["program"].eq(PROGRAM)
]
if len(program_row) != 1:
    raise RuntimeError(
        f"Expected one whitelist row for {LINEAGE}/{PROGRAM}; found {len(program_row)}"
    )
program_row = program_row.iloc[0]

draft_columns = [
    "lineage", "program", "proposed_label", "category",
    "confidence", "decision", "rationale", "top_genes",
]
display(program_row[draft_columns].to_frame("draft_value"))

current_metrics = program_metrics.loc[
    program_metrics["lineage"].eq(LINEAGE)
    & program_metrics["program"].eq(PROGRAM)
]
if len(current_metrics) != 1:
    raise RuntimeError(
        f"Expected one metric row for {LINEAGE}/{PROGRAM}; found {len(current_metrics)}"
    )
metric_columns = [
    "sd_usage", "iqr_usage", "nonzero_fraction", "n_top_sections",
    "largest_section_fraction", "effective_top_sections", "spatial_knn_enrichment",
    "spearman_nCount_Xenium", "spearman_nFeature_Xenium",
    "max_abs_pairwise_spearman_program", "max_abs_pairwise_spearman",
    "mean_top_gene_detection_fraction", "median_top_gene_log2_mean_count_ratio",
]
display(current_metrics.iloc[0][metric_columns].to_frame("condition_blind_metric"))
print(
    f"{LINEAGE}: {usage_adata.n_obs:,} cells, {len(programs)} programs, "
    f"{review_obs['blinded_sample'].nunique()} blinded sections"
)

,draft_value
lineage,excitatory_neuron
program,Usage_11
proposed_label,RELN superficial excitatory subtype
category,identity
confidence,medium
decision,include_primary
rationale,Coherent superficial neuronal program in broad...
top_genes,Reln|Cdhr1|Dner|Grik1|Slc17a7|Kcnk2|Nrn1|Stmn2


,condition_blind_metric
sd_usage,0.100668
iqr_usage,0.044911
nonzero_fraction,0.506382
n_top_sections,12.0
largest_section_fraction,0.110629
effective_top_sections,11.487994
spatial_knn_enrichment,14.560149
spearman_nCount_Xenium,0.034542
spearman_nFeature_Xenium,0.014787
max_abs_pairwise_spearman_program,Usage_6


excitatory_neuron: 235,192 cells, 25 programs, 12 blinded sections


## Usage distribution across blinded sections

Inspect the overall usage distribution and how strongly high-usage cells concentrate in individual blinded sections. Section concentration is a diagnostic flag, not an automatic exclusion criterion.

In [269]:
qc_summary = program_usage_qc_summary(
    review_obs, usage_cols=programs, top_fraction=TOP_FRACTION
)
display(qc_summary.loc[qc_summary["program"].eq(PROGRAM)])

usage_plot = review_obs[["blinded_sample", PROGRAM]].copy()
usage_plot[PROGRAM] = pd.to_numeric(usage_plot[PROGRAM], errors="coerce")
figure = px.box(
    usage_plot,
    x="blinded_sample",
    y=PROGRAM,
    points=False,
    title=f"{LINEAGE} {PROGRAM}: distribution by blinded section",
)
figure.update_layout(template="plotly_white", showlegend=False)
figure

,program,n_cells,mean_usage,median_usage,q90_usage,q95_usage,q99_usage,max_usage,top_fraction,n_top_cells,n_top_sections,largest_section_fraction,effective_top_sections
9,Usage_10,235192,0.049947,0.01112,0.14611,0.218798,0.451235,0.880199,0.05,11760,12,0.096088,11.836269


## High-usage cell audit

Review the cells that most strongly define the program. Their lineage labels, spatial domains, count depth, and detected-feature counts should be consistent with the proposed biological interpretation.

In [275]:
high_usage = select_high_usage_cells(
    review_obs, program=PROGRAM, top_fraction=TOP_FRACTION
)
display(high_usage.head(50))

high_usage_counts = (
    high_usage.groupby(
        ["blinded_sample", "cluster_sub", "spatial_domain"],
        dropna=False,
        observed=True,
    )
    .size()
    .rename("n_high_usage_cells")
    .reset_index()
    .sort_values("n_high_usage_cells", ascending=False)
)
display(high_usage_counts.head(50))

,cell_id,rank,program,usage,blinded_sample,cluster_sub,celltype_short,celltype_full,spatial_domain,nCount_Xenium,nFeature_Xenium
0,vap_46_igg:lfmlfpdm-1,1,Usage_11,0.916483,Section_11,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,265,200
1,vap_46_igg:igoblmol-1,2,Usage_11,0.908424,Section_11,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,1586,785
2,vap_50_igg:lbcnecja-1,3,Usage_11,0.894990,Section_10,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,741,488
3,naive_77_igg:gipdoejj-1,4,Usage_11,0.891780,Section_09,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,957,583
4,naive_75_igg:hlojbepb-1,5,Usage_11,0.890912,Section_03,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,467,332
5,vap_46_igg:ihholplj-1,6,Usage_11,0.881697,Section_11,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,988,600
6,vap_52_ab:jknajmfe-1,7,Usage_11,0.872393,Section_02,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,686,472
7,naive_75_igg:hknpmlha-1,8,Usage_11,0.860134,Section_03,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,1635,848
8,vap_46_igg:ihmjaofn-1,9,Usage_11,0.854155,Section_11,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,626,441
9,vap_50_igg:jnbglapa-1,10,Usage_11,0.846246,Section_10,EX-L23-IT,EX-L23-IT,Cortical Layer 2/3 Glutamatergic Neurons,8,842,519


,blinded_sample,cluster_sub,spatial_domain,n_high_usage_cells
91,Section_10,EX-L23-IT,8,1291
50,Section_06,EX-L23-IT,8,1144
71,Section_08,EX-L23-IT,8,1124
108,Section_12,EX-L23-IT,8,1068
24,Section_03,EX-L23-IT,8,1053
62,Section_07,EX-L23-IT,8,998
13,Section_02,EX-L23-IT,8,972
40,Section_05,EX-L23-IT,8,970
84,Section_09,EX-L23-IT,8,848
31,Section_04,EX-L23-IT,8,801


## Expected-gene support

Compare raw-count detection and mean expression of the program's expected genes in high-usage cells versus other cells from the same cNMF lineage. This is a coherence check, not a differential-expression test.

In [276]:
expected_genes = [gene for gene in str(program_row["top_genes"]).split("|") if gene]
gene_support = high_usage_gene_support(
    usage_adata, high_cell_ids=high_usage["cell_id"], genes=expected_genes
)
display(gene_support)
if gene_support.attrs.get("missing_genes"):
    print("Genes absent from this panel:", gene_support.attrs["missing_genes"])

,gene,high_usage_detection_fraction,background_detection_fraction,high_usage_mean_count,background_mean_count,log2_mean_count_ratio,n_high_usage_cells,n_background_cells
0,Reln,0.292772,0.0435,0.665816,0.0558,2.297303,11760,10000
1,Cdhr1,0.318282,0.0269,0.484609,0.0301,2.167851,11760,10000
2,Grik1,0.297024,0.0741,0.568793,0.1008,1.735799,11760,10000
3,Dner,0.939541,0.5612,5.783248,1.9994,1.486636,11760,10000
4,Kcnk2,0.408929,0.1794,0.670663,0.2537,1.123574,11760,10000
5,Nrn1,0.862925,0.6643,3.472449,1.5968,1.074097,11760,10000
6,Slc17a7,0.987755,0.8537,16.297534,7.9432,1.027637,11760,10000
7,Stmn2,0.929847,0.7910,5.351276,2.8561,0.882899,11760,10000


## Spatial localization

Map program usage within a blinded tissue section. Look for anatomically coherent regions or multicellular patches and check for border effects, segmentation halos, or localization driven by only a few cells.

In [182]:
high_section_counts = high_usage["blinded_sample"].value_counts()
shown_section = (
    str(high_section_counts.index[0])
    if SPATIAL_SECTION is None
    else str(SPATIAL_SECTION)
)
spatial_figure = plot_blinded_spatial_usage(
    review_obs,
    np.asarray(usage_adata.obsm["spatial"]),
    program=PROGRAM,
    blinded_sample=shown_section,
    point_size=SPATIAL_POINT_SIZE,
)
spatial_figure

## Neighboring-cell context

This table summarizes cell types near high-usage focal cells within each tissue section. Use it to distinguish an intrinsic lineage program from a pattern that may reflect neighboring-cell spillover.

In [236]:
master = ad.read_h5ad(MASTER_H5AD, backed="r")
try:
    context_ids = high_usage["cell_id"].head(MAX_CONTEXT_FOCAL_CELLS).tolist()
    neighbor_context = high_usage_neighbor_context(
        master.obs[["sample_id", "cluster_sub"]],
        np.asarray(master.obsm["spatial"]),
        focal_cell_ids=context_ids,
        sample_key="sample_id",
        cell_type_key="cluster_sub",
        radius=NEIGHBOR_RADIUS_UM,
    )
finally:
    master.file.close()
display(neighbor_context.head(40))

,neighbor_cell_type,n_neighbor_occurrences,n_focal_cells_with_neighbor,fraction_focal_cells_with_neighbor,mean_neighbors_per_focal_cell,n_focal_cells,radius
0,EX-L23-IT,10736,1988,0.9940,5.3680,2000,30.0
1,OLIG_sub1,2010,1228,0.6140,1.0050,2000,30.0
2,EC,1352,947,0.4735,0.6760,2000,30.0
3,AST_sub0,616,525,0.2625,0.3080,2000,30.0
4,MG,620,520,0.2600,0.3100,2000,30.0
5,AST_sub3,458,414,0.2070,0.2290,2000,30.0
6,OPC,506,382,0.1910,0.2530,2000,30.0
7,IN-PV,429,365,0.1825,0.2145,2000,30.0
8,IN-SST-RTN,433,328,0.1640,0.2165,2000,30.0
9,PERICYTE,331,309,0.1545,0.1655,2000,30.0


## Relationship to other programs

Program correlations provide a condition-blind view of shared or opposing usage patterns within the lineage. Use them as descriptive context alongside genes and spatial evidence.

In [228]:
correlation_obs = review_obs[programs]
if len(correlation_obs) > MAX_CORRELATION_CELLS:
    correlation_obs = correlation_obs.sample(MAX_CORRELATION_CELLS, random_state=0)

program_correlations = (
    correlation_obs.corr(method="spearman")[PROGRAM]
    .drop(PROGRAM)
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .rename_axis("other_program")
    .reset_index(name="spearman_correlation")
)
program_labels = whitelist.loc[
    whitelist["lineage"].eq(LINEAGE),
    ["program", "proposed_label", "decision"],
]
correlation_review = (
    program_correlations.merge(
        program_labels,
        left_on="other_program",
        right_on="program",
        how="left",
    )
    .drop(columns="program")
    .head(15)
)
display(correlation_review)

,other_program,spearman_correlation,proposed_label,decision
0,Usage_1,0.170147,RORB or cortical excitatory subtype,include_primary
1,Usage_5,-0.118041,Thalamic or Prkcd excitatory subtype,include_primary
2,Usage_7,-0.111341,Thalamic glutamatergic subtype,include_primary
3,Usage_21,0.108955,Immediate-early neuronal activation,include_primary
4,Usage_11,-0.095388,RELN superficial excitatory subtype,include_primary
5,Usage_12,-0.085349,Calbindin or thalamic excitatory subtype,include_primary
6,Usage_15,-0.083623,Neuronal ECM or stress-associated,include_sensitivity
7,Usage_4,0.081283,LAMP5 inhibitory-like neuronal,exclude
8,Usage_20,-0.080165,Peptidergic neuronal subtype,include_primary
9,Usage_3,0.076398,Corticothalamic or ECM-associated neuronal,include_sensitivity


## Record a review decision

Assign one of four outcomes:

- **keep_primary** — coherent and suitable for the primary feature set;
- **keep_sensitivity** — plausible but better suited to sensitivity analysis;
- **exclude** — inconsistent, technical, mixed-lineage, or likely spillover;
- **needs_followup** — insufficient evidence for admission.

Add a short evidence-based rationale. Saving remains disabled until the explicit write switch is enabled.

In [277]:
DRAFT_TO_REVIEW = {
    "include_primary": "keep_primary",
    "include_sensitivity": "keep_sensitivity",
    "exclude": "exclude",
}

REVIEW_DECISION = DRAFT_TO_REVIEW[str(program_row["decision"])]
REVIEW_CONFIDENCE = str(program_row["confidence"])
REVIEW_NOTES = ""
REVIEWER = ""
SAVE_REVIEW_DECISION = False

review_record = build_review_decision(
    lineage=LINEAGE,
    program=PROGRAM,
    review_decision=REVIEW_DECISION,
    review_confidence=REVIEW_CONFIDENCE,
    review_notes=REVIEW_NOTES,
    reviewer=REVIEWER,
    top_fraction=TOP_FRACTION,
    neighbor_radius_um=NEIGHBOR_RADIUS_UM,
    whitelist_sha256=whitelist_sha256,
)
display(pd.Series(review_record).to_frame("review_value"))

if SAVE_REVIEW_DECISION:
    saved_path = upsert_review_decision(DECISION_PATH, review_record)
    print("saved:", saved_path)
else:
    print("Not saved. Set SAVE_REVIEW_DECISION=True only after review.")

,review_value
lineage,excitatory_neuron
program,Usage_11
review_decision,keep_primary
review_confidence,high
review_notes,Coherent superficial excitatory-neuron program...
reviewer,Nicholas Rhyan
top_fraction,0.05
neighbor_radius_um,30.0
whitelist_sha256,4ecef413dd3892b2d501afe6ce60b45031225fbf6dfb6b...
reviewed_at_utc,2026-08-03T00:25:54.303993+00:00


saved: /path/to/spatial-workflow/results/ab_xenium/05_cnmf/program_review/human_review_decisions_draft.tsv


## Freeze the reviewed whitelist

Freeze the whitelist only after every admitted program has a saved decision, unresolved exclusions have been reviewed or deferred, and the decision table matches the current whitelist checksum.

The finalization script writes a versioned whitelist and manifest; only that frozen artifact should be used to construct downstream model features.